# Visual Diagnostics

Rollout comparison of a trained model against ground truth on test trajectories.
Autoregressive prediction: start from true initial state, predict 200 steps.

Set `MODEL_NAME` to `"egnn"` or `"hgnn"` and run all cells. HGNN-only panels (if any) are gated on `MODEL_NAME == "hgnn"`. Paths are anchored to the `impl/` root, so it does not matter whether Jupyter is launched from `impl/` or from `impl/evaluation/`.

In [ ]:
import sys
from pathlib import Path


def _find_impl_root() -> Path:
    """Walk up from CWD until we find the impl/ root (marked by pyproject.toml + evaluation/)."""
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").exists() and (cand / "evaluation").is_dir():
            return cand
    msg = f"could not locate impl/ root from {here}"
    raise RuntimeError(msg)


IMPL_ROOT = _find_impl_root()
if str(IMPL_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPL_ROOT))
print(f"impl root: {IMPL_ROOT}")

In [ ]:
import h5py
import torch

from evaluation.metrics import compute_single_step_metrics, run_all_rollouts
from evaluation.plots import (
    animate_trajectories,
    plot_energy,
    plot_loss_distribution,
    plot_loss_vs_distance,
    plot_pos_vel_error,
    plot_rollout_mse,
    plot_trajectories,
)
from models.egnn import EGNN
from models.hgnn import HGNN

MODEL_NAME = "egnn"  # "egnn" | "hgnn"
CKPT_PATHS = {
    "egnn": IMPL_ROOT / "runs/egnn/20260507_225609/best.pt",
    "hgnn": IMPL_ROOT / "runs/hgnn/20260507_225537/best.pt",
}
CKPT_PATH = CKPT_PATHS[MODEL_NAME]
TEST_PATH = IMPL_ROOT / "data/output/test.h5"
TRAJ_INDICES = [233, 91, 142, 266, 79]  # one per bin: close, near, mid, wide, far

device = torch.device("cpu")
ckpt = torch.load(CKPT_PATH, weights_only=False, map_location=device)
print(f"checkpoint: epoch {ckpt.epoch}, val_loss {ckpt.val_loss:.6f}")

model_cls = {"egnn": EGNN, "hgnn": HGNN}[MODEL_NAME]
model = model_cls(hidden_dim=64, n_layers=4)
model.load_state_dict(ckpt.model)
model.eval()
model.to(device)
print(f"model loaded: {MODEL_NAME.upper()}, {sum(p.numel() for p in model.parameters())} params")

with h5py.File(TEST_PATH, "r") as f:
    test_traj = f["trajectories"][:]
    test_energies = f["energies"][:]

print(f"test trajectories: {test_traj.shape}")
print(f"test energies:     {test_energies.shape}")

In [ ]:
predicted = run_all_rollouts(model, test_traj, device)
print(f"predicted: {predicted.shape}")

## 1. Trajectory rollout comparison

In [ ]:
plot_trajectories(test_traj, predicted, TRAJ_INDICES, MODEL_NAME.upper())

In [ ]:
animate_trajectories(test_traj, predicted, TRAJ_INDICES, MODEL_NAME.upper())

## 2. Energy conservation

In [ ]:
plot_energy(test_traj, predicted, TRAJ_INDICES, MODEL_NAME.upper())

## 3. Rollout MSE vs time step

In [ ]:
plot_rollout_mse(test_traj, predicted)

## 4. Position error vs velocity error

In [ ]:
plot_pos_vel_error(test_traj, predicted)

## 5. Single-step loss distribution

In [ ]:
sample_losses, min_distances = compute_single_step_metrics(model, str(TEST_PATH), device)
plot_loss_distribution(sample_losses)

## 6. Loss vs minimum pairwise distance

In [ ]:
plot_loss_vs_distance(min_distances, sample_losses)